In [ ]:
!pip install sentence-transformers chromadb groq pandas -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 93.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 159.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 105.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not curren

In [ ]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq

In [ ]:
import os
print("All libraries imported successfully")
print("Ready to build a RAG system")

All libraries imported successfully
Ready to build a RAG system


In [ ]:
GROQ_API_KEY=""
os.environ["GROQ_API_KEY"]=GROQ_API_KEY
groq_client=Groq(api_key=GROQ_API_KEY)
print("Groq api client initialized")
print("Note:if you see an authentication error later,double check your api key")

Groq api client initialized
Note:if you see an authentication error later,double check your api key


In [ ]:
df=pd.read_csv('college_notes.csv')
print("Shape of the dataset:",df.shape)
print("\nColumn Name:",df.columns.tolist())
print("\nFirst 5 rows:")
(df.head(5))

Shape of the dataset: (15, 4)

Column Name: ['note_id', 'subject', 'topic', 'content']

First 5 rows:


,note_id,subject,topic,content
0,N001,Data Engineering,ETL Pipelines,ETL stands for Extract Transform Load. It is t...
1,N002,Data Engineering,SQL Databases,A database is an organized collection of data ...
2,N003,Data Engineering,Data Cleaning,Data cleaning involves fixing or removing inco...
3,N004,Data Engineering,APIs and Data Collection,An API or Application Programming Interface al...
4,N005,Data Engineering,Big Data and PySpark,Big Data refers to extremely large datasets th...


In [ ]:
print("Subject in the dataset:")
print(df['subject'].value_counts())
print("Sample of topics:")
print(df[['note_id','subject','topic']].to_string(index=False))
print("\nLength of content(number of characters) for each notes")
df['content_length']=df['content'].apply(len)
print(df[['topic','content_length']].to_string(index=False))

Subject in the dataset:
subject
Data Engineering      5
Machine Learning      5
Generative AI         3
Python Programming    2
Name: count, dtype: int64
Sample of topics:
note_id            subject                          topic
   N001   Data Engineering                  ETL Pipelines
   N002   Data Engineering                  SQL Databases
   N003   Data Engineering                  Data Cleaning
   N004   Data Engineering       APIs and Data Collection
   N005   Data Engineering           Big Data and PySpark
   N006   Machine Learning            Supervised Learning
   N007   Machine Learning               Model Evaluation
   N008   Machine Learning            Feature Engineering
   N009   Machine Learning                 Decision Trees
   N010   Machine Learning                  Random Forest
   N011      Generative AI          Large Language Models
   N012      Generative AI             Prompt Engineering
   N013      Generative AI Retrieval Augmented Generation
   N014 Python P

#Chunking

In [ ]:
documents=df['content'].tolist()
ids=[f"note_{row['note_id']}]" for row in df.to_dict('records')]
metadatas=[
    {"subject":row['subject'],"topic":row['topic']}
    for  row in df.to_dict('records')
]
print("Total chunks prepared:",len(documents))
print("First Document ID:",ids[0])
print("First Metadata:",metadatas[0])
print("First 100 chars of document:",documents[0][:100]+"....")

Total chunks prepared: 15
First Document ID: note_N001]
First Metadata: {'subject': 'Data Engineering', 'topic': 'ETL Pipelines'}
First 100 chars of document: ETL stands for Extract Transform Load. It is the process of collecting raw data from different sourc....


In [ ]:
#Embedding
print("Loading Embedding model")
print("This may take 30-60 seconds on first run model is being downloaded")
print("Subsequent runs will be faster as the model is cached")

embedding_model=SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model loaded successfully!!")
test_embedding=embedding_model.encode('This is a test sentence')
print("Test embedding shape:",test_embedding.shape)
print("First 5 values of test document:",test_embedding[:5])

Loading Embedding model
This may take 30-60 seconds on first run model is being downloaded
Subsequent runs will be faster as the model is cached


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!!
Test embedding shape: (384,)
First 5 values of test document: [0.0715524  0.06848021 0.00660334 0.10176961 0.0111223 ]


In [ ]:
chroma_client=chromadb.Client()
collection=chroma_client.get_or_create_collection('colleg_notes_rag')
print("Chroma client created!")
print("Collection name:college_notes_rag")
print("Documents in collection so far:",collection.count())

Chroma client created!
Collection name:college_notes_rag
Documents in collection so far: 0


In [ ]:
print("Generating embeddings for all 15 notes")
print("This may take 15-30 seconds")
embeddings=embedding_model.encode(documents,show_progress_bar=True)
print("Embedding matrix shape:",embeddings.shape)
collection.add(
    documents=documents,
    embeddings=embeddings,
    ids=ids,
    metadatas=metadatas
)


Generating embeddings for all 15 notes
This may take 15-30 seconds


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding matrix shape: (15, 384)


In [ ]:
def retrieve_chunks(question,top_k=3):
  """Given a user question,retreive the most relevent document chunks from chromadb.
  parameters:
  question(str):the users's question as a text string
  top_k(int):How many top results to return (default:3)
  returns:
  a dictionary containing retrived documents,distances,metadata"""

  question_embedding=embedding_model.encode(question).tolist()
  results=collection.query(
     query_embeddings=question_embedding,
     n_results=top_k
  )
  return results
print("Retrivel function defined sucessfully")
print("functions retrive_relevent_chunks(metadata,top_k=3)")

Retrivel function defined sucessfully
functions retrive_relevent_chunks(metadata,top_k=3)


In [ ]:
test_question="what is ETL and how does it work in data engineering"
print(f"test question:{test_question}")
print("="*60)

results =retrieve_chunks(test_question, top_k=3)

print("\top 3 retrived chunks")
print("="*60)

for i,(doc,dist,meta) in enumerate(zip(results['documents'][0],results['distances'][0],results['metadatas'][0])):
  print(f"results:{i+1}")
  print(f"subject:{meta['subject']}")
  print(f"topic:{meta['topic']}")
  print(f"distance:{dist:.4f}")
  print(f"content:{doc[:120]}...")

test question:what is ETL and how does it work in data engineering
	op 3 retrived chunks
results:1
subject:Data Engineering
topic:ETL Pipelines
distance:0.2041
content:ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it i...
results:2
subject:Data Engineering
topic:APIs and Data Collection
distance:1.1100
content:An API or Application Programming Interface allows two software applications to talk to each other. In data engineering ...
results:3
subject:Python Programming
topic:Data Visualization
distance:1.3892
content:Data visualization is the process of representing data as charts graphs and visual formats. Python libraries like Matplo...


#Context Injection


> Rag prompt template
You are a helpful  academic assistant.



In [ ]:
def build_context_from_results(results):
  """Format ChromaDB retreival results into a readable context string.
  Parameters:
     Results:The output from collection.query()-a dictionary
  Returns:
     context_str(str): A formatted string of all retrieved documents"""
  context_parts=[]
  for i,(doc,meta)in enumerate(zip(results['documents'][0],results['metadatas'][0])):
    chunk_test=f"[Source:{i+1}+{meta['subject']}+{meta['topic']}\n{doc}]"
    context_parts.append(chunk_test)
    context_str="\n\n---\n\n".join(context_parts)
    return context_str
context=build_context_from_results(results)
print("Built context string from retrieved chunks:")
print("="*60)
print(context[:500]+'.....')
print("Total context length:",len(context),"characters")


Built context string from retrieved chunks:
[Source:1+Data Engineering+ETL Pipelines
ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehouse for analysis.].....
Total context length: 258 characters


In [ ]:
def generate_rag_answer(question, context):
  system_prompt = """You are a helpful academic assistant for engineering students,=.

  You will be given context retrieved from a college knowledge base, and a student's question.

  RULES:
  1. Answer ONLY using the information provided in the context below.
  2. If the answer is not found in the context, say exactly:
  "I don't have enough information in my knowledge base to answer this question."
  3. Do not use your general training knowledge.
  4. Keep answers clear, accurate, and beginner-friendly.
  5. Mention which source the information came from when possible."""
  user_prompt = f"""Context from Knowledge Base:{context}
---
   Studnets's Question: {question}

Please answer the question based only on the context provided above."""

  response=groq_client.chat.completions.create(
      model="llama-3.1-8b-instant",
      messages=[
          {"role":"system","content":system_prompt},
          {"role":"user","content":user_prompt}
      ],
      temperature=0.1,
      max_tokens=500
  )
  answer=response.choices[0].message.content
  return answer
print("RAG generation function defined")


RAG generation function defined


In [ ]:
import os
#Build the complete end to end pipeline
def ask_college_assistant(question,top_k=3,verbose=True):
  """Ask the college assistant a question and print the results"""

  if verbose:
    print("Question:",question)
    print("="*60)
    print("Step 1: Retrieving most relevant documents")
  results=retrieve_chunks(question,top_k=3)
  if verbose:
    print("Retrieved:",top_k,"chunks from the knowledge base")
    for i,meta in enumerate(results['metadatas'][0]):
      print(f"{i+1}.{meta['subject']}-{meta['topic']}")
    print("\nStep 2:Building context string....")
  context=build_context_from_results(results)

  if verbose:
    print("Context Build",len(context),"characters")
    print("Step 3:Sending to LLM for answer generation")
  answer=generate_rag_answer(question,context)

  if verbose:
    print("\n"+"="*60)
    print("ANSWER")
    print("="*60)
    print(answer)
    print("="*60)
  return answer
print("Complete RAG pipeline defined")
print("Function:ask_college_assistant(question,top_k=3")

Complete RAG pipeline defined
Function:ask_college_assistant(question,top_k=3


In [ ]:
question_1="What is ETL and what are its three main stages?"
answer_1=ask_college_assistant(question_1,top_k=3,verbose=True)

Question: What is ETL and what are its three main stages?
Step 1: Retrieving most relevant documents
Retrieved: 3 chunks from the knowledge base
1.Data Engineering-ETL Pipelines
2.Generative AI-Retrieval Augmented Generation
3.Generative AI-Prompt Engineering

Step 2:Building context string....
Context Build 258 characters
Step 3:Sending to LLM for answer generation

ANSWER
ETL stands for Extract Transform Load. 

Its three main stages are:

1. Extract: Collecting raw data from different sources.
2. Transform: Transforming the raw data into a clean and structured format.
3. Load: Loading the transformed data into a database or data warehouse for analysis.

[Source: 1+Data Engineering+ETL Pipelines]


In [ ]:
question_2="How do embeddings help in building search system?"
answer_2=ask_college_assistant(question_2,top_k=3,verbose=True)

Question: How do embeddings help in building search system?
Step 1: Retrieving most relevant documents
Retrieved: 3 chunks from the knowledge base
1.Generative AI-Retrieval Augmented Generation
2.Generative AI-Large Language Models
3.Machine Learning-Feature Engineering

Step 2:Building context string....
Context Build 330 characters
Step 3:Sending to LLM for answer generation

ANSWER
I don't have enough information in my knowledge base to answer this question.


In [ ]:
question_3="What is the population of Tokyo?"
answer_3=ask_college_assistant(question_3,top_k=3,verbose=True)

Question: What is the population of Tokyo?
Step 1: Retrieving most relevant documents
Retrieved: 3 chunks from the knowledge base
1.Generative AI-Large Language Models
2.Data Engineering-SQL Databases
3.Data Engineering-Data Cleaning

Step 2:Building context string....
Context Build 273 characters
Step 3:Sending to LLM for answer generation

ANSWER
I don't have enough information in my knowledge base to answer this question.


In [ ]:
def retrieve_by_subject(question,subject_filter,top_k=3):
  """Retrieve  relevant chunks but only from """
  question_embedding=embedding_model.encode(question).tolist()
  results=collection.query(
      query_embeddings=question_embedding,
      n_results=top_k,
      where={"subject":subject_filter}
  )
  return results

print("Retrieving only from GENAI subject")
print("="*60)

filtered_results=retrieve_by_subject(
      question="how do LLM'S generate text?",
      subject_filter="GenAI",
      top_k=3)
for i,(doc,meta) in enumerate(zip(filtered_results['documents'][0],filtered_results['metadatas'][0],)):
    print("Result",i+1,[[meta['subject']],meta['topic']])
    print(doc[:100]+"...")

Retrieving only from GENAI subject
